# Step 0 — Regime Engine Validation Gate
Walk-forward validation of the 6-cell regime classifier (direction x vol) per
`projects/research/regime-dashboard-plan.md`. **Gate:** if cells do not separate
forward return/vol distributions, the dashboard does not get built.

Dual-mode: tries live data (yfinance + vix-utils); if unreachable, falls back
to synthetic regime-switching data for STRUCTURAL verification only (no evidence value).
**Designed to run in Google Colab** — Runtime > Run all. Also runs locally unchanged.

In [ ]:
# Environment setup — installs deps on Colab, no-op elsewhere
import importlib.util, subprocess, sys
IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "yfinance", "vix-utils", "hmmlearn", "nest_asyncio"], check=True)
print("colab:", IN_COLAB)

In [ ]:
import os, warnings, json, math
from pathlib import Path
from itertools import permutations
import numpy as np
import pandas as pd
from scipy import stats as sstats
from scipy.stats import norm
import matplotlib
if not IN_COLAB:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
np.random.seed(7)
print("imports ok | numpy", np.__version__, "| pandas", pd.__version__)

In [ ]:
# Parameters (papermill-injectable)
if IN_COLAB:
    DATA_DIR = Path("/content/data"); OUTPUT_DIR = Path("/content/output")
else:
    DATA_DIR = Path("../../data"); OUTPUT_DIR = Path("../../output")   # vault-relative: projects/research/
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_MODE = "auto"          # "auto" | "live" | "synthetic"
START = "2005-01-01"
N_STATES = 3                # direction states
REFIT_EVERY = 63            # trading days between HMM refits
MIN_TRAIN = 756             # min obs before first fit (3y)
SEP_Z_THRESH = 0.0          # DIAGNOSTIC ONLY as of v3 — logged per refit, does NOT block
                            # acceptance (see v3 note below). z = gap between adjacent
                            # sorted state means / pooled standard error (each state's own
                            # variance + occupancy count). v2 used this as a hard accept/
                            # reject gate at 1.5 and it backfired: both synthetic AND live
                            # SPY data show median z ~0.9, so a 1.5 bar rejected 63/64 live
                            # refits and froze the model on its very first fit — which for
                            # START=2005 landed on 2005-2007, a quiet pre-GFC window, then
                            # got filtered across 2008/2020/2022 unchanged. That produced
                            # worse damage (test5 worst-case drawdown -15% vs -4% baseline)
                            # than the v1 bug it replaced. Root cause of v1 (sort-based
                            # label swaps) is fixed by continuity matching below on its
                            # own — the separation gate was redundant belt-and-suspenders
                            # that turned out to be the dominant failure mode. Kept as a
                            # tunable knob (raise above 0 to reintroduce blocking) once the
                            # real z-distribution is better understood, but defaults off.
COMMIT_P, COMMIT_DAYS, MIN_DWELL = 0.70, 2, 3
VOL_W = dict(vix_pct=1.2, backwardation=1.0, rv_pct=1.0, bias=-1.6)  # logistic weights
COND_WINDOW, MIN_COND_OBS = 252, 40  # v5: direction-conditional vol percentile window/floor
                            # (trailing same-direction-label OBSERVATIONS, not calendar days)
CURVE_K_GRID = [0.0, 0.15, 0.30, 0.60]  # forecast tilt sensitivities
FWD_HORIZONS = [5, 21]
RISK_FREE = 0.03
TRADE_DTE = 21              # trading days to expiry for structure backtest
TRADE_STEP = 5              # entry every N days
print("params ok")

In [ ]:
# Data acquisition: live (yfinance + vix-utils) with synthetic fallback
def try_live():
    import yfinance as yf
    import vix_utils
    try:
        import nest_asyncio; nest_asyncio.apply()   # vix-utils sync loader inside Jupyter's event loop
    except ImportError:
        pass
    px = yf.download(["SPY", "^VIX", "^VIX3M", "^VIX9D"], start=START,
                     auto_adjust=True, progress=False)["Close"]
    px = px.rename(columns={"^VIX": "VIX", "^VIX3M": "VIX3M", "^VIX9D": "VIX9D"})
    ts = vix_utils.load_vix_term_structure()
    wide = vix_utils.pivot_futures_on_monthly_tenor(ts)
    vx = pd.DataFrame({"VX1": wide[1]["Close"], "VX3": wide[3]["Close"]})
    vx.index = pd.to_datetime(vx.index)
    df = px.join(vx, how="inner").dropna(subset=["SPY", "VIX", "VX1", "VX3"])
    return df

def make_synthetic(n=2800):
    # 3 direction x 2 vol hidden states; VX slope correlated with vol state and
    # predictive of direction transitions (structural test only, NOT evidence)
    mu = {0: -0.0010, 1: 0.0001, 2: 0.0007}
    sg = {0: 0.016, 1: 0.009, 2: 0.007}
    dates = pd.bdate_range("2015-01-01", periods=n)
    d_state, v_state = 2, 0
    rows = []
    slope = 0.08
    for t in range(n):
        slope = 0.9 * slope + 0.1 * (0.10 if v_state == 0 else -0.06) + np.random.normal(0, 0.02)
        p_bear = 0.006 + max(0, -slope) * 0.55
        if d_state == 2 and np.random.rand() < p_bear: d_state = 1
        elif d_state == 1 and np.random.rand() < (0.03 + max(0, -slope) * 0.6):
            d_state = 0 if np.random.rand() < 0.5 else 2
        elif d_state == 0 and np.random.rand() < 0.035: d_state = 1
        if np.random.rand() < 0.02: v_state = 1 - v_state
        vol_mult = 1.0 if v_state == 0 else 2.3
        r = np.random.normal(mu[d_state], sg[d_state] * vol_mult)
        rows.append((dates[t], r, d_state, v_state, slope))
    sim = pd.DataFrame(rows, columns=["date", "ret", "true_dir", "true_vol", "slope_raw"]).set_index("date")
    df = pd.DataFrame(index=sim.index)
    df["SPY"] = 100 * np.exp(sim["ret"].cumsum())
    rv = sim["ret"].rolling(5).std().bfill() * np.sqrt(252)
    df["VIX"] = 100 * rv * (1 + 0.15 * np.random.randn(n) * 0.1) + 4
    df["VX1"] = df["VIX"] * (1 + np.where(sim["slope_raw"] > 0, 0.01, -0.01))
    df["VX3"] = df["VX1"] * (1 + sim["slope_raw"])
    df["VIX3M"] = df["VIX"] * (1 + sim["slope_raw"] * 0.8)
    df["VIX9D"] = df["VIX"] * (1 - sim["slope_raw"] * 0.3)
    df.attrs["truth"] = sim[["true_dir", "true_vol"]]
    return df

MODE = DATA_MODE
if DATA_MODE == "auto":
    try:
        df = try_live(); MODE = "live"
    except Exception as e:
        print(f"[!] live pull failed ({type(e).__name__}: {str(e)[:90]}) -> synthetic fallback")
        df = make_synthetic(); MODE = "synthetic"
elif DATA_MODE == "live":
    df = try_live()
else:
    df = make_synthetic()
banner = "LIVE DATA — results are evidence" if MODE == "live" else \
         "SYNTHETIC DATA — STRUCTURAL RUN ONLY, ZERO EVIDENCE VALUE"
print("=" * 70 + f"\n  {banner}\n" + "=" * 70)
print(df.shape, df.index.min().date(), "->", df.index.max().date())

In [ ]:
# Feature construction (all causal)
feat = pd.DataFrame(index=df.index)
feat["ret"] = np.log(df["SPY"]).diff()
feat["rv20"] = feat["ret"].rolling(20).std() * np.sqrt(252)
feat["slope"] = (df["VX3"] - df["VX1"]) / df["VX1"]          # VX1-VX3 curve slope (canonical)
feat["slope_z"] = (feat["slope"] - feat["slope"].rolling(252).mean()) / feat["slope"].rolling(252).std()
feat["vix"] = df["VIX"]
feat["vix_pct"] = feat["vix"].rolling(252).rank(pct=True)
feat["rv_pct"] = feat["rv20"].rolling(252).rank(pct=True)
feat["backwardation"] = (feat["slope"] < 0).astype(float)
feat = feat.dropna(subset=["ret", "rv20", "slope_z", "vix_pct", "rv_pct"])
print("features:", feat.shape)

In [ ]:
# Direction engine: walk-forward 3-state Gaussian HMM, causal filtered posterior
#
# LABEL-CONTINUITY FIX (post step-0 v1 failure): independently sorting each refit's
# states by raw mean caused label swaps across refits when two state means were close
# (common — daily-return means are tiny relative to noise), which spliced into a
# sign-inverted 21d forward-return effect. Fix: (1) match each new fit's states to the
# PREVIOUS fit's canonical means via nearest-assignment (not a fresh sort), so "bull"
# stays "bull" across windows; (2) if the fitted means aren't well separated
# (min gap < MIN_GAP_SD x return std), skip the relabel/refit entirely and keep using
# the previous model for that window rather than accept a noisy ordering.
from hmmlearn.hmm import GaussianHMM

def fit_hmm(r, prev_means_ordered=None, z_thresh=SEP_Z_THRESH):
    m = GaussianHMM(n_components=N_STATES, covariance_type="diag",
                    n_iter=200, random_state=7, min_covar=1e-6)
    m.fit(r.reshape(-1, 1))
    raw_means = m.means_.ravel()
    raw_vars = np.array([m.covars_[i].ravel()[0] for i in range(N_STATES)])
    state_seq = m.predict(r.reshape(-1, 1))
    n_i = np.array([max((state_seq == i).sum(), 1) for i in range(N_STATES)])  # occupancy per state
    ord_by_mean = np.argsort(raw_means)
    sm, sv, sn = raw_means[ord_by_mean], raw_vars[ord_by_mean], n_i[ord_by_mean]
    se = np.sqrt(sv[:-1] / sn[:-1] + sv[1:] / sn[1:])                          # SE of adjacent mean gap
    z = np.diff(sm) / np.maximum(se, 1e-12)
    separated = bool(np.all(z > z_thresh))                                     # every adjacent pair must clear it
    if prev_means_ordered is None:
        order = ord_by_mean                                # first fit: mean-sort defines bear/neutral/bull
    else:
        best_perm, best_cost = None, np.inf                # continuity: nearest-previous-mean assignment
        for perm in permutations(range(N_STATES)):
            cost = sum(abs(raw_means[perm[i]] - prev_means_ordered[i]) for i in range(N_STATES))
            if cost < best_cost:
                best_cost, best_perm = cost, perm
        order = np.array(best_perm)
    return m, order, raw_means, separated, float(z.min())

def forward_filter(r, m, order):
    means = m.means_.ravel()[order]
    stds = np.sqrt(np.array([m.covars_[i].ravel()[0] for i in range(N_STATES)]))[order]
    A = m.transmat_[np.ix_(order, order)]
    pi = m.startprob_[order]
    alphas = np.zeros((len(r), N_STATES))
    lik = norm.pdf(r[:, None], means[None, :], stds[None, :]) + 1e-300
    a = pi * lik[0]; a /= a.sum(); alphas[0] = a
    for t in range(1, len(r)):
        a = (A.T @ a) * lik[t]; a /= a.sum(); alphas[t] = a
    return alphas, A

r = feat["ret"].values
n = len(r)
post = np.full((n, N_STATES), np.nan)
Amats = {}
last_fit, prev_means_ordered = None, None
n_refit, n_skipped, z_hist = 0, 0, []
for t in range(MIN_TRAIN, n, REFIT_EVERY):
    m, order, raw_means, separated, zmin = fit_hmm(r[:t], prev_means_ordered)
    z_hist.append(zmin)
    # v3: ALWAYS accept the continuity-matched refit — the z-guard's job (blocking noisy
    # relabels) is superseded by the nearest-previous-mean assignment in fit_hmm() itself,
    # which prevents label-order swaps regardless of separation quality. Blocking on z
    # (v2) froze the model on a single stale early fit instead. z is still tracked/logged
    # below purely as a per-window confidence diagnostic (SEP_Z_THRESH>0 would re-enable
    # gating if a future run shows a genuine need for it).
    if separated:
        n_refit += 1
    else:
        n_skipped += 1                                     # diagnostic count only, not acted on
    prev_means_ordered = m.means_.ravel()[order]
    seg_end = min(t + REFIT_EVERY, n)
    alphas, A = forward_filter(r[:seg_end], m, order)
    post[t:seg_end] = alphas[t:seg_end]
    Amats[t] = A
    last_fit = t
dirpost = pd.DataFrame(post, index=feat.index, columns=["p_bear", "p_neut", "p_bull"]).dropna()
print("HMM filtered posteriors:", dirpost.shape, "| windows:", n_refit + n_skipped,
      f"| above-z-thresh={n_refit} below={n_skipped} (diagnostic only, all refits accepted)",
      f"| median z-separation={np.median(z_hist):.2f}")
print(dirpost.mean().round(3).to_dict())

In [ ]:
# Curve-conditioned Bayesian drift model: mu_t = a + b*slope_z, conjugate NIG, walk-forward OOS
H = 5
sl = feat["slope_z"].reindex(dirpost.index)
fwd = feat["ret"].reindex(feat.index).rolling(H).sum().shift(-H).reindex(dirpost.index)
p_up = pd.Series(np.nan, index=dirpost.index)
b_mean = pd.Series(np.nan, index=dirpost.index)
X_all = np.column_stack([np.ones(len(sl)), sl.values])
y_all = fwd.values
for i in range(252, len(sl)):
    m_idx = slice(0, max(0, i - H))               # only matured pairs
    X, y = X_all[m_idx], y_all[m_idx]
    ok = ~np.isnan(y); X, y = X[ok], y[ok]
    if len(y) < 100: continue
    # NIG posterior with vague prior
    V0inv = np.eye(2) * 1e-4
    Vn = np.linalg.inv(V0inv + X.T @ X)
    bn = Vn @ (X.T @ y)
    a_n = 1.0 + len(y) / 2
    b_n = 1.0 + 0.5 * (y @ y - bn @ np.linalg.inv(Vn) @ bn)
    s2 = b_n / a_n
    x0 = X_all[i]
    mu_pred = x0 @ bn
    var_pred = s2 * (1 + x0 @ Vn @ x0)
    p_up.iloc[i] = 1 - sstats.t.cdf(0, df=2 * a_n, loc=mu_pred, scale=np.sqrt(var_pred))
    b_mean.iloc[i] = bn[1]
drift_dir = pd.cut(p_up, [-np.inf, 0.35, 0.65, np.inf], labels=[0, 1, 2]).astype(float)
print("drift model OOS coverage:", p_up.notna().mean().round(3),
      "| mean beta(slope):", np.nanmean(b_mean).round(5))

In [ ]:
# Vol layer: logistic score, causal features only
# v5 NOTE (REVERTED): direction-conditional vol percentiles were tried here to remove a
# hypothesized leverage-effect confound (bear days baseline more volatile, so an
# unconditional threshold makes bull_hi a rarer anomaly than bear_hi). Tested live: it did
# NOT shrink the bull_hi-vs-bear_hi reversal (SPY 21d effect: -1.08sd before, -1.08sd
# after, unchanged) and it COST backtest performance (Sharpe 2.93->2.64, hit rate
# 66%->55%). Cross-sectionally several names got a STRONGER reversal after conditioning,
# not weaker -- evidence the reversal is a genuine momentum-exhaustion property of
# trailing-return HMM states, not a vol-threshold artifact. Documented negative result;
# reverted to the original unconditional design below. See TEST 1d for the fix that's
# actually being tested now (VX curve state, an information source orthogonal to returns).
vf = feat.reindex(dirpost.index)
z = (VOL_W["vix_pct"] * vf["vix_pct"] + VOL_W["backwardation"] * vf["backwardation"]
     + VOL_W["rv_pct"] * vf["rv_pct"] + VOL_W["bias"])
p_high = 1 / (1 + np.exp(-4 * z))
print("P(high vol): mean", p_high.mean().round(3), "| >0.5 share", (p_high > 0.5).mean().round(3))

In [ ]:
# Commitment: raw cell posterior -> smoothed committed regime (0.70 x 2d + 3d dwell)
cells = ["bear_hi", "bear_lo", "neut_hi", "neut_lo", "bull_hi", "bull_lo"]
cp = pd.DataFrame(index=dirpost.index, columns=cells, dtype=float)
for i, d in enumerate(["p_bear", "p_neut", "p_bull"]):
    cp[cells[2 * i]] = dirpost[d] * p_high
    cp[cells[2 * i + 1]] = dirpost[d] * (1 - p_high)
raw_cell = cp.idxmax(axis=1)
committed = []
cur, streak, dwell = None, 0, 0
for t in range(len(cp)):
    top = raw_cell.iloc[t]; ptop = cp.iloc[t].max()
    if cur is None:
        cur = top
    else:
        if top != cur and ptop >= COMMIT_P:
            streak += 1
        else:
            streak = 0
        if streak >= COMMIT_DAYS and dwell >= MIN_DWELL:
            cur, streak, dwell = top, 0, 0
    dwell += 1
    committed.append(cur)
committed = pd.Series(committed, index=cp.index, name="regime")
print(committed.value_counts().to_dict())

In [ ]:
# Regime ribbon over price
fig, ax = plt.subplots(figsize=(13, 4))
spy = df["SPY"].reindex(committed.index)
colors = {"bull_lo": "#2e7d32", "bull_hi": "#a5d6a7", "neut_lo": "#9e9e9e", "neut_hi": "#e0e0e0",
          "bear_lo": "#ef9a9a", "bear_hi": "#c62828"}
for cell in cells:
    mask = committed == cell
    ax.fill_between(committed.index, 0, 1, where=mask, transform=ax.get_xaxis_transform(),
                    color=colors[cell], alpha=0.35, label=cell)
ax.plot(spy.index, spy.values, "k-", lw=0.8)
ax.set_yscale("log"); ax.legend(ncol=6, fontsize=7); ax.set_title(f"Committed regime ribbon [{MODE}]")
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "step0_ribbon.png", dpi=110); plt.show()
print("ribbon saved")

In [ ]:
# TEST 1 — forward return / vol separation by committed cell
res1 = {}
tbl = []
for h in FWD_HORIZONS:
    fr = feat["ret"].reindex(committed.index).rolling(h).sum().shift(-h)
    fv = feat["ret"].reindex(committed.index).rolling(h).std().shift(-h) * np.sqrt(252)
    g = pd.DataFrame({"cell": committed, "fr": fr, "fv": fv}).dropna()
    stats_h = g.groupby("cell").agg(n=("fr", "size"), fwd_ret_mu=("fr", "mean"),
                                    fwd_ret_sd=("fr", "std"), fwd_vol_mu=("fv", "mean"))
    stats_h["ann_ret"] = stats_h["fwd_ret_mu"] * 252 / h
    tbl.append(stats_h.assign(h=h))
    bull = g.loc[g.cell.str.startswith("bull"), "fr"]
    bear = g.loc[g.cell.str.startswith("bear"), "fr"]
    ks = sstats.ks_2samp(bull, bear) if len(bull) > 30 and len(bear) > 30 else None
    es = (bull.mean() - bear.mean()) / g["fr"].std() if len(bear) > 0 else np.nan
    res1[h] = dict(ks_p=float(ks.pvalue) if ks else None, effect=float(es))
    print(f"h={h}d  bull-vs-bear KS p={res1[h]['ks_p']:.2e}  effect={es:.2f}sd" if ks else f"h={h}d insufficient")
sep_table = pd.concat(tbl)
print(sep_table.round(4).to_string())

In [ ]:
# TEST 1b — soft (posterior-weighted) forward-return check, diagnostic against TEST 1
# Uses the raw 6-cell posterior (cp, pre-smoothing/pre-argmax) as continuous weights
# instead of the hard committed label. If TEST 1's hard-label effect flips sign or
# vanishes here, the issue is in labeling/argmax/smoothing, not in the underlying
# state separation itself.
res1b = {}
for h in FWD_HORIZONS:
    fr = feat["ret"].reindex(cp.index).rolling(h).sum().shift(-h)
    ok = fr.notna()
    w_bull = cp.loc[ok, ["bull_hi", "bull_lo"]].sum(1)
    w_bear = cp.loc[ok, ["bear_hi", "bear_lo"]].sum(1)
    bull_soft = (w_bull * fr[ok]).sum() / w_bull.sum()
    bear_soft = (w_bear * fr[ok]).sum() / w_bear.sum()
    es_soft = (bull_soft - bear_soft) / fr[ok].std()
    res1b[h] = dict(bull_soft=float(bull_soft), bear_soft=float(bear_soft), effect_soft=float(es_soft))
    print(f"h={h}d  soft bull-mean={bull_soft:.4f}  soft bear-mean={bear_soft:.4f}  "
          f"effect(soft)={es_soft:.2f}sd  [hard was {res1[h]['effect']:.2f}sd]")

In [ ]:
# TEST 1c — vol-conditional separation (bull vs bear WITHIN each vol bucket) + omnibus
# Rationale: three live runs now reproduce the SAME large negative marginal bull-vs-bear
# effect at 21d (-1.32sd, -1.30sd, -1.32sd) with the HMM machinery behaving identically
# (n_refit=64, median z=0.90 each time) -- this rules out label-swap/freeze noise as the
# cause. The first live run's per-cell table showed bull_hi had strongly negative 21d
# forward returns (melt-up / blow-off-top behavior) while bull_lo was positive (genuine
# grind-up trend continuing) -- economically coherent, and exactly the reason direction
# and vol are modeled as separate axes. TEST 1's marginal comparison collapses both vol
# buckets into one "bull" group before comparing to "bear," which can produce a
# misleading sign flip even when each vol bucket separates sensibly on its own. Testing
# within-bucket (and an omnibus across all 6 cells) is the criterion that actually
# matches how the structure matrix consumes these cells -- it never acts on direction
# alone, always on direction-x-vol.
res1c = {"pairs": {}, "omnibus": {}}
for h in FWD_HORIZONS:
    fr = feat["ret"].reindex(committed.index).rolling(h).sum().shift(-h)
    g = pd.DataFrame({"cell": committed, "fr": fr}).dropna()
    for vb in ["hi", "lo"]:
        bull = g.loc[g.cell == f"bull_{vb}", "fr"]; bear = g.loc[g.cell == f"bear_{vb}", "fr"]
        key = f"{h}_{vb}"
        if len(bull) > 30 and len(bear) > 30:
            ks = sstats.ks_2samp(bull, bear)
            es = (bull.mean() - bear.mean()) / g["fr"].std()
            res1c["pairs"][key] = dict(n_bull=int(len(bull)), n_bear=int(len(bear)),
                                       ks_p=float(ks.pvalue), effect=float(es))
            print(f"h={h}d vol={vb}  bull n={len(bull)} bear n={len(bear)}  "
                  f"KS p={ks.pvalue:.2e}  effect={es:.2f}sd")
        else:
            res1c["pairs"][key] = dict(n_bull=int(len(bull)), n_bear=int(len(bear)), ks_p=None, effect=None)
            print(f"h={h}d vol={vb}  insufficient (n_bull={len(bull)} n_bear={len(bear)})")
    groups = [g.loc[g.cell == c, "fr"] for c in cells if (g.cell == c).sum() > 20]
    if len(groups) >= 3:
        kw = sstats.kruskal(*groups)
        res1c["omnibus"][str(h)] = float(kw.pvalue)
        print(f"h={h}d  Kruskal-Wallis across {len(groups)} cells: p={kw.pvalue:.2e}")
    else:
        res1c["omnibus"][str(h)] = None

In [ ]:
# TEST 1d — curve-gating: does VX1-VX3 term structure disambiguate breakout vs relief
# rally within bull_hi/bull_lo? Hypothesis: a bull cell entered while the curve is
# BACKWARDATED (VX1>VX3, near-term fear still priced despite the price bounce) behaves
# like a relief rally and mean-reverts; entered in CONTANGO (curve normalized) it behaves
# like genuine continuation. Unlike v5's vol-threshold conditioning (reverted -- didn't
# help, cost backtest Sharpe), this uses an information source ORTHOGONAL to the return
# series the HMM itself is fit on, so it isn't subject to the same null result.
res1d = {}
curve_sign = feat["slope"].reindex(committed.index).apply(
    lambda s: "backwardation" if s < 0 else ("contango" if pd.notna(s) else None))
for h in FWD_HORIZONS:
    fr = feat["ret"].reindex(committed.index).rolling(h).sum().shift(-h)
    g = pd.DataFrame({"cell": committed, "fr": fr, "curve": curve_sign}).dropna()
    for dcell in ["bull_hi", "bull_lo"]:
        sub = g[g.cell == dcell]
        cont = sub.loc[sub.curve == "contango", "fr"]; back = sub.loc[sub.curve == "backwardation", "fr"]
        key = f"{h}_{dcell}"
        if len(cont) > 30 and len(back) > 30:
            ks = sstats.ks_2samp(cont, back)
            es = (cont.mean() - back.mean()) / sub["fr"].std()
            res1d[key] = dict(n_contango=int(len(cont)), n_backwardation=int(len(back)),
                              mean_contango=float(cont.mean()), mean_backwardation=float(back.mean()),
                              ks_p=float(ks.pvalue), effect=float(es))
            confirmed = "CONFIRMED" if (ks.pvalue < 0.05 and es > 0) else "not confirmed"
            print(f"h={h}d {dcell}: contango n={len(cont)} mean={cont.mean():.4f}  "
                  f"backwardation n={len(back)} mean={back.mean():.4f}  "
                  f"KS p={ks.pvalue:.2e}  effect(contango-backwardation)={es:.2f}sd  [{confirmed}]")
        else:
            res1d[key] = dict(n_contango=int(len(cont)), n_backwardation=int(len(back)), ks_p=None, effect=None)
            print(f"h={h}d {dcell}: insufficient (contango={len(cont)}, backwardation={len(back)})")

In [ ]:
# TEST 2 — transition matrices + direction x vol independence
lag = committed.shift(1).dropna(); cur6 = committed.loc[lag.index]
T6 = pd.crosstab(lag, cur6, normalize="index")
d_map = committed.str.split("_").str[0]; v_map = committed.str.split("_").str[1]
Td = pd.crosstab(d_map.shift(1).dropna(), d_map.loc[d_map.index[1:]], normalize="index")
Tv = pd.crosstab(v_map.shift(1).dropna(), v_map.loc[v_map.index[1:]], normalize="index")
# independence check: joint switch prob vs product of marginals
joint_switch = ((d_map != d_map.shift(1)) & (v_map != v_map.shift(1))).mean()
indep_pred = (d_map != d_map.shift(1)).mean() * (v_map != v_map.shift(1)).mean()
print("P(both axes switch same day) actual={:.4f} vs independence={:.4f}  ratio={:.1f}x".format(
    joint_switch, indep_pred, joint_switch / max(indep_pred, 1e-9)))
print("\n6-cell persistence (diag):", np.round(np.diag(T6.reindex(index=T6.columns).fillna(0)), 3))
res2 = dict(joint_switch=float(joint_switch), indep_pred=float(indep_pred))

In [ ]:
# TEST 3 — flip-flop audit: switches with vs without smoothing
sw_raw = int((raw_cell != raw_cell.shift(1)).sum())
sw_com = int((committed != committed.shift(1)).sum())
yrs = len(committed) / 252
print(f"switches/yr  raw={sw_raw / yrs:.1f}  committed={sw_com / yrs:.1f}  reduction={1 - sw_com / max(sw_raw,1):.0%}")
res3 = dict(raw_per_yr=sw_raw / yrs, committed_per_yr=sw_com / yrs)

In [ ]:
# TEST 4 — curve-conditioning value: next-day direction forecast, static vs curve-tilted
truth_next = dirpost.values[1:].argmax(1)
slope_z = feat["slope_z"].reindex(dirpost.index).values[:-1]
fit_keys = sorted(Amats)
res4 = {}
for k in CURVE_K_GRID:
    hits, lls = [], []
    for t in range(len(truth_next)):
        gi = [x for x in fit_keys if x <= t + MIN_TRAIN]
        A = Amats[gi[-1] if gi else fit_keys[0]].copy()
        if k > 0 and np.isfinite(slope_z[t]):
            w = np.exp(np.array([-1, 0, 1]) * k * slope_z[t])   # contango boosts bull, backwardation boosts bear
            A = A * w[None, :]; A = A / A.sum(1, keepdims=True)
        f = dirpost.values[t] @ A
        hits.append(f.argmax() == truth_next[t])
        lls.append(np.log(f[truth_next[t]] + 1e-12))
    res4[k] = dict(hit=float(np.mean(hits)), ll=float(np.mean(lls)))
    print(f"k={k:0.2f}  next-day hit={res4[k]['hit']:.4f}  logscore={res4[k]['ll']:.4f}")
best_k = max(res4, key=lambda k: res4[k]["ll"])
print("best tilt k:", best_k, "| TVTP (phase 2b) justified:" , best_k > 0)
# drift R^2 check: slope-conditioned vs intercept-only (OOS sign hit)
ok = p_up.notna() & fwd.notna()
sign_hit = ((p_up[ok] > 0.5) == (fwd[ok] > 0)).mean()
print(f"drift model OOS sign hit-rate: {sign_hit:.3f} (naive long baseline: {(fwd[ok] > 0).mean():.3f})")
res4["drift_sign_hit"] = float(sign_hit)

In [ ]:
# TEST 5 — approximate structure-matrix backtest (BS pricing, realized-vol proxy) vs baselines
def bs(S, K, T, sig, r_, cp):
    if T <= 0: return max(cp * (S - K), 0.0)
    d1 = (np.log(S / K) + (r_ + 0.5 * sig**2) * T) / (sig * np.sqrt(T))
    d2 = d1 - sig * np.sqrt(T)
    if cp == 1: return S * norm.cdf(d1) - K * np.exp(-r_ * T) * norm.cdf(d2)
    return K * np.exp(-r_ * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

def k_from_delta(S, T, sig, r_, delta, cp):
    if cp == 1: d1 = norm.ppf(delta)
    else: d1 = norm.ppf(1 + delta)                 # put delta negative convention
    return S * np.exp(-(d1 * sig * np.sqrt(T) - (r_ + 0.5 * sig**2) * T))

MATRIX = {  # cell -> (structure, legs) legs: list of (cp, delta, +1 long/-1 short)
    "bull_hi": ("short_put", [(-1, -0.30, -1)]),
    "bull_lo": ("bull_call_spread", [(1, 0.55, 1), (1, 0.30, -1)]),
    "bear_hi": ("bear_call_spread", [(1, 0.30, -1), (1, 0.15, 1)]),
    "bear_lo": ("bear_put_spread", [(-1, -0.55, 1), (-1, -0.30, -1)]),
    "neut_hi": ("wide_bull_put_spread", [(-1, -0.20, -1), (-1, -0.10, 1)]),
    "neut_lo": ("no_trade", []),
}
# TEST 5c — bear_hi redesign: bear_hi's forward returns are unusually STRONG (test1c: bear_hi
# beats bull_hi by ~1.08sd at 21d) -- the ORIGINAL bear_call_spread bets against that. Candidate
# fix per discussion with Nikolas: a 1x2 call ratio backspread (sell 1 near-the-money call,
# buy 2 further OTM) -- defined/bounded max loss (worst point is at the long strike, not
# unlimited), open-ended upside if the historical bounce-off-capitulation pattern plays out.
# Because bear_hi is BY DEFINITION high vol, the near-ATM short call is relatively rich,
# which should make this cheaper to enter than in a calm regime. Tests the SAME caveat as
# curve-gating: likely a handful of autocorrelated episodes (2008-09, 2020, 2022), so this
# is validated via defined-risk structure choice, not by trading it with confidence in the
# sample size.
MATRIX_V2 = dict(MATRIX)
MATRIX_V2["bear_hi"] = ("call_backspread_1x2", [(1, 0.40, -1), (1, 0.20, 2)])
spy_s = df["SPY"].reindex(committed.index)
sig_s = (feat["rv20"].reindex(committed.index) * 1.1).clip(0.05, 2.0)
slope_s = feat["slope"].reindex(committed.index)   # TEST 5b: curve-gated variant
T0 = TRADE_DTE / 252
pnl_matrix, pnl_always, pnl_gated, pnl_v2, dates_tr = [], [], [], [], []
bearhi_old, bearhi_new = [], []   # isolated bear_hi-only comparison (paired, same dates)
for t in range(0, len(committed) - TRADE_DTE, TRADE_STEP):
    S0, sg0 = spy_s.iloc[t], sig_s.iloc[t]
    ST = spy_s.iloc[t + TRADE_DTE]
    if not (np.isfinite(S0) and np.isfinite(sg0) and np.isfinite(ST)): continue
    def trade(legs):
        p = 0.0
        for cp_, dlt, pos in legs:
            K = k_from_delta(S0, T0, sg0, RISK_FREE, dlt, 1 if cp_ == 1 else 0) if cp_ == 1 else \
                k_from_delta(S0, T0, sg0, RISK_FREE, dlt, 0)
            entry = bs(S0, K, T0, sg0, RISK_FREE, cp_)
            exitv = bs(ST, K, 1e-9, sg0, RISK_FREE, cp_)
            p += pos * (exitv - entry)
        return p / S0                                # normalize by spot
    cell_t = committed.iloc[t]
    _, legs = MATRIX[cell_t]
    pnl_matrix.append(trade(legs) if legs else 0.0)
    pnl_always.append(trade([(-1, -0.30, -1), (-1, -0.15, 1)]))   # always-on bull put spread
    # TEST 5b: skip bull_hi/bull_lo entries when the curve is backwardated at entry
    # (relief-rally signature per TEST 1d) -- the concrete gating rule TEST 1d validates
    slope_t = slope_s.iloc[t]
    gated_legs = [] if (cell_t in ("bull_hi", "bull_lo") and pd.notna(slope_t) and slope_t < 0) else legs
    pnl_gated.append(trade(gated_legs) if gated_legs else 0.0)
    # TEST 5c: same trade stream, but bear_hi uses the call backspread (MATRIX_V2) instead
    # of the original bear_call_spread. Every other cell is identical to `matrix`.
    _, legs_v2 = MATRIX_V2[cell_t]
    pnl_v2.append(trade(legs_v2) if legs_v2 else 0.0)
    if cell_t == "bear_hi":
        bearhi_old.append(trade(legs) if legs else 0.0)
        bearhi_new.append(trade(legs_v2) if legs_v2 else 0.0)
    dates_tr.append(committed.index[t])
bt = pd.DataFrame({"matrix": pnl_matrix, "matrix_curve_gated": pnl_gated,
                   "matrix_bearhi_backspread": pnl_v2, "always_bps": pnl_always}, index=dates_tr)
def sharpe(x): return x.mean() / (x.std() + 1e-12) * np.sqrt(252 / TRADE_STEP)
summary5 = pd.DataFrame({"mean_per_trade": bt.mean(), "sharpe_ann": bt.apply(sharpe),
                         "hit": (bt > 0).mean(), "worst": bt.min()})
print(summary5.round(4).to_string())
res5 = summary5.to_dict()
gate_helps = res5["sharpe_ann"]["matrix_curve_gated"] > res5["sharpe_ann"]["matrix"]
print(f"curve-gating {'IMPROVES' if gate_helps else 'does NOT improve'} on the ungated matrix "
      f"(Sharpe {res5['sharpe_ann']['matrix_curve_gated']:.2f} vs {res5['sharpe_ann']['matrix']:.2f})")
v2_helps = res5["sharpe_ann"]["matrix_bearhi_backspread"] > res5["sharpe_ann"]["matrix"]
print(f"bear_hi call backspread {'IMPROVES' if v2_helps else 'does NOT improve'} on the aggregate "
      f"(Sharpe {res5['sharpe_ann']['matrix_bearhi_backspread']:.2f} vs {res5['sharpe_ann']['matrix']:.2f})")
# Isolated, paired comparison restricted to bear_hi trades only -- the aggregate Sharpe above
# mixes in every other unchanged cell, which can dilute or mask the effect specific to this swap.
bh_old, bh_new = np.array(bearhi_old), np.array(bearhi_new)
res5c_isolated = {"n_bearhi_trades": int(len(bh_old))}
if len(bh_old) > 5:
    res5c_isolated.update(
        mean_old=float(bh_old.mean()), mean_new=float(bh_new.mean()),
        hit_old=float((bh_old > 0).mean()), hit_new=float((bh_new > 0).mean()),
        worst_old=float(bh_old.min()), worst_new=float(bh_new.min()),
        best_old=float(bh_old.max()), best_new=float(bh_new.max()),
    )
    print(f"\nbear_hi ONLY (n={len(bh_old)} trades) -- old (bear_call_spread) vs new (call_backspread_1x2):")
    print(f"  mean/trade: {bh_old.mean():.4f} -> {bh_new.mean():.4f}")
    print(f"  hit rate:   {(bh_old>0).mean():.3f} -> {(bh_new>0).mean():.3f}")
    print(f"  worst:      {bh_old.min():.4f} -> {bh_new.min():.4f}")
    print(f"  best:       {bh_old.max():.4f} -> {bh_new.max():.4f}")
else:
    print(f"\nbear_hi ONLY: insufficient trades in this backtest window (n={len(bh_old)})")

In [ ]:
# VERDICT — automated gate summary
verdict = {
    "mode": MODE,
    "test1_separation_MARGINAL_diagnostic_only": res1,   # collapses vol axis -- NOT the gate criterion, see test1c
    "test1b_soft_diagnostic": res1b,
    "test1c_vol_conditional_separation": res1c,           # PRIMARY separation criterion
    "test1d_curve_gating": res1d,                         # breakout-vs-relief-rally test
    "hmm_refit_stats": {"n_refit": n_refit, "n_skipped_low_separation": n_skipped,
                        "median_z_separation": float(np.median(z_hist))},
    "test2_independence": res2,
    "test3_flipflop": res3,
    "test4_curve": {str(k): v for k, v in res4.items() if not isinstance(v, float)},
    "test4_drift_sign_hit": res4.get("drift_sign_hit"),
    "test5_backtest": {k: {kk: float(vv) for kk, vv in v.items()} for k, v in res5.items()},
    "test5c_bearhi_backspread_isolated": res5c_isolated,
}
gate = None
if MODE == "live":
    # PRIMARY separation criterion (v4): does bull separate from bear WITHIN at least one
    # vol bucket, or do the 6 cells differ at all (omnibus)? Marginal bull-vs-bear
    # (collapsing vol) is logged above for transparency but no longer gates -- three live
    # runs showed it can flip sign from vol-bucket-mixing alone (see test1c rationale).
    pair_hits = [v for v in res1c["pairs"].values()
                 if v.get("ks_p") is not None and v["ks_p"] < 0.01 and v.get("effect") is not None and v["effect"] > 0.10]
    omni_hits = [p for p in res1c["omnibus"].values() if p is not None and p < 0.01]
    sep_ok = bool(pair_hits) or bool(omni_hits)
    curve_ok = max(res4[k]["ll"] for k in CURVE_K_GRID if isinstance(res4.get(k), dict) and k > 0) > res4[0.0]["ll"]
    edge_ok = res5["sharpe_ann"]["matrix"] > res5["sharpe_ann"]["always_bps"]
    gate = "PASSED" if (sep_ok and edge_ok) else "FAILED"
    verdict["gate"] = dict(separation=sep_ok, curve_conditioning_adds=curve_ok,
                           matrix_beats_alwayson=edge_ok, GATE=gate,
                           note="separation now judged on test1c (vol-conditional/omnibus), not marginal test1")
    print("GATE:", gate, "|", verdict["gate"])
else:
    verdict["gate"] = "N/A — synthetic structural run; execute on quant box with live data"
    print(verdict["gate"])
with open(OUTPUT_DIR / "step0_verdict.json", "w") as fh:
    json.dump(verdict, fh, indent=2, default=str)
print("verdict written ->", OUTPUT_DIR / "step0_verdict.json")
if IN_COLAB:
    try:
        from google.colab import files
        files.download(str(OUTPUT_DIR / "step0_verdict.json"))
    except Exception as e:
        print("(auto-download skipped:", e, "- grab it from the Files pane)")